In [1]:
# Notebook 2 — Fine-tune PhoBERT cho bài toán khôi phục dấu tiếng Việt
# Notebook này độc lập với Notebook 1 nhưng tải lại đúng corpus 100k câu
# (cùng code, cùng seed 42) nên train/val/test giống hệt Notebook 1.
# Công thức bài toán: phân loại token — mỗi từ không dấu được xếp vào 1
# MẪU DẤU (pattern): các ký tự có dấu tại những vị trí có thể mang dấu
# của từ đó. Vd "được" -> mẫu "đượ", "có" -> "ó", "hoàn" -> "oà".
# Cùng một mẫu có nghĩa giống nhau cho mọi từ nên model học được quy
# luật dấu từ ngữ cảnh (xem Cell 2).
# Cập nhật 17/11 (review code): thêm assert kiểm tra khớp dữ liệu (Cell 1);
# Cell 4 đánh giá val mỗi epoch và giữ checkpoint tốt nhất theo eval loss.
# Số PhoBERT trong báo cáo (83,4%) đo ngày 16/11 bằng model epoch cuối,
# trước các sửa đổi này — chạy lại có thể lệch nhẹ.

# Cell 1 — Hàm bỏ dấu + tải Wikipedia + chia train/val/test
import unicodedata, re, random, collections

def strip_diacritics(s: str) -> str:
    # Bỏ dấu tiếng Việt: tách dấu bằng NFD rồi loại bỏ ký tự dấu,
    # riêng đ/Đ xử lý tay vì NFD không tách được.
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn")

from datasets import load_dataset

N_ARTICLES = 10000
TARGET_SENTENCES = 100000

ds = load_dataset("wikimedia/wikipedia", "20231101.vi",
                  split="train", streaming=True)
sentences, seen = [], set()
n_read = 0
for ex in ds:
    n_read += 1
    if n_read > N_ARTICLES:
        break
    text = ex["text"].replace("\n", " ")
    for s in re.split(r"(?<=[.!?…])\s+(?=[A-ZÀ-ỸĐ])", text):
        s = re.sub(r"\s+", " ", s).strip()
        w = s.split()
        if not (5 <= len(w) <= 40):
            continue
        if "http" in s or s.count("=") > 2:
            continue
        if sum(c.isdigit() for c in s) / max(len(s), 1) > 0.15:
            continue
        if s in seen:
            continue
        seen.add(s)
        sentences.append(s)
        if len(sentences) >= TARGET_SENTENCES:
            break
    if len(sentences) >= TARGET_SENTENCES:
        break
print(f"bai viet da doc: {n_read} | cau giu lai: {len(sentences)}")

random.seed(42)
# Chuẩn hóa câu về NFC: Wikipedia có chỗ lưu dấu dạng tách rời (NFD,
# vd "ê" = "e" + dấu mũ tách riêng) khiến từ có dấu và từ không dấu
# lệch độ dài. NFC gộp mỗi ký tự có dấu thành 1 ký tự duy nhất.
pairs = [(strip_diacritics(s), unicodedata.normalize("NFC", s))
         for s in sentences]
random.shuffle(pairs)
n = len(pairs)
n_train, n_val = int(n * 0.8), int(n * 0.1)
train = pairs[:n_train]
val = pairs[n_train:n_train + n_val]
test = pairs[n_train + n_val:]
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

# Kiểm tra khớp với Notebook 1: tổng số từ phải = 2.387.449 —
# nếu khác thì tập dữ liệu đã lệch.
words = [w for _, acc in pairs for w in acc.split()]
print(f"tong so tu: {len(words)}")
# Dừng ngay nếu tập dữ liệu lệch khỏi Notebook 1.
assert len(words) == 2387449, f"tap du lieu lech: {len(words)} tu (muon 2.387.449)"

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

bai viet da doc: 1501 | cau giu lai: 100000
train 80000 | val 10000 | test 10000
tong so tu: 2387449


In [2]:
# Cell 2 — Mã hóa dấu thành "mẫu dấu" (signature)
# Thay vì bắt model chọn 1 trong các dạng có dấu của từng từ (nhãn khác
# nhau cho mỗi từ, model khó học), ta tách riêng PHẦN DẤU ra khỏi từ.
# Mẫu dấu của một từ = tuple các đơn vị có dấu tại những vị trí có thể
# mang dấu, theo đúng thứ tự. Ví dụ: "được" -> ("đ", "ư", "ợ");
# "có" -> ("ó",); "hoàn" -> ("o", "à") (o giữ nguyên, a->à).
# Từ không có vị trí mang dấu (vd "123", "Mr") có mẫu rỗng ().
# MODIFIABLE để CỐ ĐỊNH theo tiếng Việt (nguyên âm a e i o u y + đ),
# không học từ dữ liệu: Wikipedia chứa chữ nước ngoài (Thái, Khmer,
# Hy Lạp...) nên học từ dữ liệu làm số mẫu dấu bùng nổ tới hàng chục
# nghìn và model học kém. Tên riêng nước ngoài chứa dấu phụ âm ngoài
# tiếng Việt (vd "Rêš", "François") thì không mã hóa được bằng mẫu
# dấu tiếng Việt — chúng chiếm rất ít (0,03%) và được bỏ qua khi train.
# Mẫu dấu lưu dạng tuple vì Wikipedia có chỗ lưu dấu dạng tách rời
# (vd "ê" = "e" + dấu mũ riêng): tách từ thành đơn vị (ký tự gốc +
# dấu đi kèm) bằng NFD rồi gộp mỗi đơn vị về NFC thì mã hóa đúng
# với mọi dạng lưu trữ.
from collections import Counter, defaultdict

# Vị trí có thể mang dấu trong tiếng Việt (cố định, không học từ data)
MODIFIABLE = set("aeiouyAEIOUYdD")

def split_units(word):
    # Tách từ thành các đơn vị: mỗi đơn vị = 1 ký tự gốc + dấu đi kèm.
    # Vd "Rês" -> ["R", "ê"] dù "ê" lưu dạng gộp hay tách rời.
    units = []
    for ch in unicodedata.normalize("NFD", word):
        if unicodedata.category(ch) == "Mn" and units:
            units[-1] += ch  # dấu đi kèm ký tự trước nó
        else:
            units.append(ch)
    # Gộp mỗi đơn vị về NFC cho gọn (vd "e" + dấu mũ -> "ê")
    return [unicodedata.normalize("NFC", u) for u in units]

def apply_signature(bare, sig):
    # Khôi phục từ có dấu từ từ không dấu + mẫu dấu.
    # Mẫu phải có đúng số đơn vị = số vị trí có thể mang dấu.
    chars = list(bare)
    pos = [i for i, b in enumerate(bare) if b in MODIFIABLE]
    if len(sig) != len(pos):
        return None  # mẫu không khớp từ này
    for i, s in zip(pos, sig):
        chars[i] = s
    return "".join(chars)

def to_signature(bare, acc):
    # Mẫu dấu = tuple các đơn vị có dấu tại vị trí có thể mang dấu.
    # Trả về None nếu từ không mã hóa được bằng mẫu dấu tiếng Việt:
    # số đơn vị khác số ký tự gốc, hoặc khôi phục lại không khớp gold
    # (vd tên riêng nước ngoài chứa dấu phụ âm ngoài tiếng Việt).
    units = split_units(acc)
    if len(units) != len(bare):
        return None
    sig = tuple(u for u, b in zip(units, bare) if b in MODIFIABLE)
    got = apply_signature(bare, sig)
    if got is None or unicodedata.normalize("NFC", got) != unicodedata.normalize("NFC", acc):
        return None
    return sig

# Chẩn đoán: đếm từ chưa ở dạng NFC (Cell 1 đã chuẩn hóa NFC nên
# con số này phải = 0; nếu > 0 nghĩa là bước chuẩn hóa chưa chạy).
n_non_nfc = sum(1 for _, acc in train for a in acc.split()
                if unicodedata.normalize("NFC", a) != a)
print(f"so tu chua o dang NFC trong train: {n_non_nfc}")

# Thống kê mã hóa trên toàn bộ train: từ không mã hóa được (chủ yếu
# tên riêng nước ngoài) sẽ mang nhãn -100 khi train và được tính là
# lỗi khi đánh giá — tỉ lệ của chúng phải rất nhỏ.
n_total = n_unencodable = 0
unenc_examples = []
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        n_total += 1
        if to_signature(b, a) is None:
            n_unencodable += 1
            if len(unenc_examples) < 5:
                unenc_examples.append((b, a))
print(f"tong so tu train: {n_total} | khong ma hoa duoc: "
      f"{n_unencodable} ({n_unencodable / n_total:.3%})")
for b, a in unenc_examples:
    print(f"  vd khong ma hoa duoc: {b!r} -> {a!r}")
assert n_unencodable / n_total < 0.01, "ti le tu khong ma hoa duoc qua cao"

sig_counts = Counter()
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        sig = to_signature(b, a)
        if sig is not None:
            sig_counts[sig] += 1
# Tập nhãn = các mẫu dấu xuất hiện từ 5 lần trở lên trong train.
# Phần lớn mẫu chỉ xuất hiện 1-2 lần (đuôi dài của từ hiếm) — giữ
# chúng thì số lớp bùng lên hàng nghìn mà model không đủ ví dụ để
# học; bỏ chúng đi thì các từ đó mang nhãn -100 khi train (không
# đóng góp vào loss), khi dự đoán model chọn mẫu tương thích phổ
# biến nhất. Không có mẫu nào bị mất khỏi dữ liệu, chỉ không làm nhãn.
MIN_SIG_COUNT = 5
signatures = sorted(s for s, c in sig_counts.items() if c >= MIN_SIG_COUNT)
sig_to_id = {s: i for i, s in enumerate(signatures)}
NUM_LABELS = len(signatures)
n_dropped = sum(1 for c in sig_counts.values() if c < MIN_SIG_COUNT)
n_dropped_tok = sum(c for c in sig_counts.values() if c < MIN_SIG_COUNT)
print(f"so mau dau khac nhau (giu lai): {NUM_LABELS}")
print(f"so mau bi loai (xuat hien < {MIN_SIG_COUNT} lan): {n_dropped} | "
      f"so tu bi anh huong: {n_dropped_tok}")
print("10 mau pho bien nhat:",
      [("".join(s), c) for s, c in sig_counts.most_common(10)])

# Với mỗi ký tự gốc, tập các đơn vị có dấu đã thấy (để khóa các mẫu
# không tương thích khi dự đoán, vd mẫu ("ó",) không dùng được cho "ba").
valid_forms = defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        units = split_units(a)
        if len(units) != len(b):
            continue
        for bc, unit in zip(b, units):
            if bc in MODIFIABLE:
                valid_forms[bc].add(unit)

# Tập từ đa nghĩa (dùng để đo accuracy riêng, giống Notebook 1)
forms = defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        forms[b].add(a)
amb_set = {b for b, v in forms.items() if len(v) >= 2}
print(f"tu da nghia: {len(amb_set)}")

so tu chua o dang NFC trong train: 0
tong so tu train: 1909902 | khong ma hoa duoc: 545 (0.029%)
  vd khong ma hoa duoc: 'Res' -> 'Rêš'
  vd khong ma hoa duoc: 'Francois' -> 'François'
  vd khong ma hoa duoc: 'Sulba' -> 'Śulba'
  vd khong ma hoa duoc: 'Svetasvatara-Upanisad,' -> 'Śvetāśvatara-Upaniṣad,'
  vd khong ma hoa duoc: 'Spis.' -> 'Spiš.'
so mau dau khac nhau (giu lai): 2326
so mau bi loai (xuat hien < 5 lan): 6840 | so tu bi anh huong: 10255
10 mau pho bien nhat: [('à', 99813), ('á', 73418), ('', 57010), ('a', 43325), ('o', 42296), ('ô', 35566), ('ó', 32465), ('ộ', 31375), ('i', 30733), ('ủa', 28972)]
tu da nghia: 5461


In [3]:
# Cell 3 — Tokenize bằng tokenizer của PhoBERT, gắn nhãn mẫu dấu
# Tokenizer của PhoBERT là tokenizer Python (chậm) nên không có word_ids().
# Cách căn nhãn thay thế: tokenize từng từ riêng (từ sau từ đầu tiên được
# thêm khoảng trắng ở đầu để giữ đúng tiền tố Ġ của BPE), rồi ghép lại.
# Với tokenizer kiểu RoBERTa, BPE không bao giờ gộp qua ranh giới từ nên
# cách này cho ra đúng dãy subword như khi tokenize cả câu — cell có bước
# kiểm chứng ở dưới (so sánh 200 câu mẫu).
# Chỉ subword ĐẦU TIÊN của mỗi từ mang nhãn; các subword còn lại và
# token đặc biệt mang nhãn -100 (Trainer sẽ bỏ qua khi tính loss).
# Từ mà mẫu dấu đúng chưa từng xuất hiện trong train cũng mang nhãn -100.
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
print("tokenizer nhanh:", tokenizer.is_fast)

_sub_cache = {}
def word_subtokens(word, first):
    # Subword của 1 từ; từ không phải đầu câu thêm " " ở đầu để BPE
    # tạo đúng tiền tố Ġ như khi từ đó đứng trong câu.
    # Cache theo từ vì cùng 1 từ lặp lại rất nhiều lần trong corpus.
    key = (word, first)
    if key not in _sub_cache:
        _sub_cache[key] = tokenizer.tokenize(word if first else " " + word)
    return _sub_cache[key]

def tokenize_pairs(pairs):
    all_input_ids, all_attn, all_labels = [], [], []
    n_unseen_sig = 0
    for bare, acc in pairs:
        bare_words, gold_words = bare.split(), acc.split()
        input_ids = [tokenizer.cls_token_id]
        labels = [-100]  # token [CLS] không tính loss
        for i, b in enumerate(bare_words):
            sub = word_subtokens(b, first=(i == 0))
            ids = tokenizer.convert_tokens_to_ids(sub)
            input_ids.extend(ids)
            # Nhãn = id của mẫu dấu đúng
            sig = to_signature(b, gold_words[i])
            if sig in sig_to_id:
                labels.append(sig_to_id[sig])
            else:
                labels.append(-100)  # mẫu dấu chưa thấy trong train
                n_unseen_sig += 1
            labels.extend([-100] * (len(ids) - 1))
        input_ids.append(tokenizer.sep_token_id)
        labels.append(-100)  # token [SEP] không tính loss
        # Cắt ngắn nếu câu quá dài (hiếm: câu tối đa 40 từ)
        if len(input_ids) > 128:
            input_ids, labels = input_ids[:128], labels[:128]
        all_input_ids.append(input_ids)
        all_attn.append([1] * len(input_ids))
        all_labels.append(labels)
    print(f"tu co mau dau chua thay trong train: {n_unseen_sig}")
    return {"input_ids": all_input_ids,
            "attention_mask": all_attn, "labels": all_labels}

# Kiểm chứng: 200 câu mẫu — dãy subword ghép từ từng từ phải giống hệt
# dãy subword khi tokenize cả câu. Nếu khác, dừng lại (assert) thay vì
# train trên dữ liệu căn nhãn sai.
import random as _random
# Dùng Random(0) cục bộ thay vì seed lại module random toàn cục
# (tránh làm lệch trạng thái RNG của các cell sau).
for bare, _ in _random.Random(0).sample(train, 200):
    sent_tokens = tokenizer.tokenize(bare)
    word_tokens = []
    for i, w in enumerate(bare.split()):
        word_tokens.extend(word_subtokens(w, first=(i == 0)))
    assert sent_tokens == word_tokens, f"lech tokenize: {bare}"
print("kiem chung tokenize: 200/200 cau khop")

train_ds = Dataset.from_dict(tokenize_pairs(train))
val_ds = Dataset.from_dict(tokenize_pairs(val))
test_ds = Dataset.from_dict(tokenize_pairs(test))
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

tokenizer nhanh: False
kiem chung tokenize: 200/200 cau khop
tu co mau dau chua thay trong train: 10800
tu co mau dau chua thay trong train: 1307
tu co mau dau chua thay trong train: 1429
train 80000 | val 10000 | test 10000


In [4]:
# Cell 4 — Fine-tune PhoBERT (token classification, nhãn = mẫu dấu)
# PhoBERT-base đã pretrain sẵn trên ~20GB tiếng Việt; ở đây chỉ thêm
# đầu phân loại (số lớp = số mẫu dấu) và train trên dữ liệu của đề án.
# Đây là fine-tune, KHÔNG phải pretrain lại từ đầu.
import inspect
import torch
import transformers
from transformers import (AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)

print("transformers:", transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| so GPU:", torch.cuda.device_count())

# Bảo vệ: nếu số mẫu dấu quá lớn (>= 10000) thì công thức mã hóa có
# vấn đề (mẫu gần như trùng cả từ, như lần học MODIFIABLE từ dữ liệu
# cho ra 42.525 mẫu) — dừng lại thay vì train vô ích. Mẫu dấu là
# tuple các đơn vị nguyên âm nên từ có nhiều nguyên âm tạo ra hàng
# nghìn tổ hợp; sau khi chỉ giữ mẫu xuất hiện >= 5 lần thì số mẫu
# còn vài nghìn — đó là mức bình thường.
assert NUM_LABELS < 10000, f"so mau dau qua lon: {NUM_LABELS}"

model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base", num_labels=NUM_LABELS)

# Tham số huấn luyện. Tên tham số có thể khác nhau giữa các phiên bản
# transformers nên chỉ giữ lại những tham số mà TrainingArguments hiện
# tại chấp nhận, các tham số còn lại in ra để biết chứ không làm lỗi cell.
accepted = set(inspect.signature(TrainingArguments).parameters)
# Tên tham số đánh giá khác nhau giữa các bản transformers (5.x dùng
# eval_strategy, 4.x dùng evaluation_strategy) — chọn tên mà bản hiện
# tại chấp nhận. Chỉ bật load_best_model_at_end khi có đánh giá val
# (không thì Trainer báo lỗi).
if "eval_strategy" in accepted:
    eval_kw = {"eval_strategy": "epoch"}
elif "evaluation_strategy" in accepted:
    eval_kw = {"evaluation_strategy": "epoch"}
else:
    eval_kw = {}
wanted = dict(
    output_dir="/kaggle/working/phobert-run",
    num_train_epochs=3,              # 3 epoch cho công thức mẫu dấu
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,              # lr nhỏ vì chỉ fine-tune
    # warmup 750 bước ≈ 10% tổng số bước (7.500). Dùng warmup_steps thay
    # warmup_ratio vì transformers 5.20.2 trên Kaggle không có warmup_ratio.
    warmup_steps=750,
    weight_decay=0.01,
    logging_steps=200,
    # Đánh giá val mỗi epoch và giữ checkpoint tốt nhất theo eval loss
    # (trước 17/11 eval_strategy mặc định "no" nên val không được dùng
    # và model epoch cuối được lấy một cách mù quáng).
    **eval_kw,
    save_strategy="epoch",
    load_best_model_at_end=bool(eval_kw),
    report_to="none",
    fp16=True,                       # mixed precision cho GPU T4
)
args = TrainingArguments(
    **{k: v for k, v in wanted.items() if k in accepted})
dropped = [k for k in wanted if k not in accepted]
print("tham so bi bo (phien ban nay khong co):", dropped or "khong co")

collator = DataCollatorForTokenClassification(tokenizer, padding=True)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator)
trainer.train()

# Lưu model tốt nhất (load_best_model_at_end đã nạp checkpoint tốt nhất
# vào model) + tokenizer để dùng lại ở Cell 5 và demo sau này
trainer.save_model("/kaggle/working/phobert-model")
tokenizer.save_pretrained("/kaggle/working/phobert-model")
print("da luu model vao /kaggle/working/phobert-model")

transformers: 5.20.2
device: cuda | so GPU: 2


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tham so bi bo (phien ban nay khong co): ['warmup_ratio']


... (2 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,12.391943
400,8.359326
600,5.948574
800,4.613728
1000,3.798919
1200,3.311873
1400,2.969352
1600,2.720800
1800,2.542931
2000,2.411852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

da luu model vao /kaggle/working/phobert-model


In [5]:
# Cell 5 — Đánh giá PhoBERT trên tập test (cùng 3 metric như baseline)
# Với mỗi từ, model cho điểm từng mẫu dấu; chỉ xét các mẫu TƯƠNG THÍCH
# với từ đó (đúng độ dài + từng ký tự là dạng có dấu hợp lệ của ký tự
# gốc), lấy argmax rồi khôi phục từ có dấu bằng apply_signature.
# Nhờ học mẫu dấu mà model đoán được cả từ chưa từng thấy trong train.
import csv, torch
from collections import defaultdict

# Bảng tra cứu: bộ khung (các ký tự tại vị trí có thể mang dấu) ->
# danh sách id các mẫu dấu tương thích. Tính 1 lần để dự đoán nhanh.
def skeleton(bare):
    return tuple(b for b in bare if b in MODIFIABLE)

by_len = defaultdict(list)
for i, s in enumerate(signatures):
    by_len[len(s)].append(i)

all_skeletons = {skeleton(b) for bare, _ in test for b in bare.split()}
valid_ids = {}
for skel in all_skeletons:
    valid_ids[skel] = [i for i in by_len[len(skel)] if all(
        signatures[i][j] in valid_forms[skel[j]]
        for j in range(len(skel)))]
print(f"so bo khung khac nhau trong test: {len(valid_ids)}")

model.eval()
BATCH = 64
pad_id = tokenizer.pad_token_id
pred_sents = []
with torch.no_grad():
    for start in range(0, len(test), BATCH):
        chunk = test[start:start + BATCH]
        # Tokenize từng câu: [CLS] + subword từng từ + [SEP],
        # đồng thời ghi lại vị trí subword đầu tiên của mỗi từ.
        batch_ids, batch_first_pos = [], []
        for bare, _ in chunk:
            ids = [tokenizer.cls_token_id]
            first_pos = []
            for i, w in enumerate(bare.split()):
                first_pos.append(len(ids))
                ids.extend(tokenizer.convert_tokens_to_ids(
                    word_subtokens(w, first=(i == 0))))
            ids.append(tokenizer.sep_token_id)
            batch_ids.append(ids[:128])
            batch_first_pos.append([p for p in first_pos if p < 128])
        maxlen = max(len(x) for x in batch_ids)
        input_ids = torch.tensor([x + [pad_id] * (maxlen - len(x))
                                  for x in batch_ids], device=device)
        attn = torch.tensor([[1] * len(x) + [0] * (maxlen - len(x))
                             for x in batch_ids], device=device)
        logits = model(input_ids=input_ids,
                       attention_mask=attn).logits.cpu()
        for i, (bare, _) in enumerate(chunk):
            bare_words = bare.split()
            sent_pred = []
            for wi, b in enumerate(bare_words):
                if wi >= len(batch_first_pos[i]):
                    sent_pred.append(b)  # từ bị cắt do câu quá dài
                    continue
                ids = valid_ids[skeleton(b)]
                if not ids:
                    sent_pred.append(b)  # không có mẫu nào tương thích
                    continue
                j = batch_first_pos[i][wi]
                lg = logits[i, j]
                best = max(ids, key=lambda k: lg[k].item())
                restored = apply_signature(b, signatures[best])
                sent_pred.append(restored if restored is not None else b)
            pred_sents.append(" ".join(sent_pred))

# Đo 3 metric giống hệt Notebook 1 (Cell 6/Cell 8)
tot = cor = sent_tot = sent_cor = amb_tot = amb_cor = 0
for (bare, acc), pred in zip(test, pred_sents):
    pw, aw = pred.split(), acc.split()
    sent_tot += 1
    sent_cor += (pred == acc)
    for p, a, b in zip(pw, aw, bare.split()):
        tot += 1
        cor += (p == a)
        if b in amb_set:
            amb_tot += 1
            amb_cor += (p == a)

print(f"phobert — word accuracy: {cor}/{tot} = {cor / tot:.1%}")
print(f"phobert — sentence accuracy: {sent_cor}/{sent_tot} = {sent_cor / sent_tot:.1%}")
if amb_tot:
    print(f"phobert — accuracy tren tu da nghia: {amb_cor}/{amb_tot} = "
          f"{amb_cor / amb_tot:.1%}")

# Accuracy riêng trên từ chưa từng thấy trong train (điểm mạnh của
# mô hình học mẫu dấu: nó đoán được dấu cho từ mới, còn từ điển thì không)
seen_words = set(forms)
un_tot = un_cor = 0
for (bare, acc), pred in zip(test, pred_sents):
    for p, a, b in zip(pred.split(), acc.split(), bare.split()):
        if b not in seen_words:
            un_tot += 1
            un_cor += (p == a)
if un_tot:
    print(f"accuracy tren tu chua thay trong train: {un_cor}/{un_tot} = "
          f"{un_cor / un_tot:.1%}")

for (bare, acc), pred in list(zip(test, pred_sents))[:3]:
    print(f"  IN : {bare}\n  OUT: {pred}\n  GOLD: {acc}")

# Lưu dự đoán để chặng so sánh dùng lại
with open("/kaggle/working/phobert_predictions.csv", "w",
          encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["bare", "gold", "pred"])
    for (bare, acc), pred in zip(test, pred_sents):
        w.writerow([bare, acc, pred])
print("da luu phobert_predictions.csv")

so bo khung khac nhau trong test: 2143
phobert — word accuracy: 198894/239189 = 83.2%
phobert — sentence accuracy: 629/10000 = 6.3%
phobert — accuracy tren tu da nghia: 169293/207644 = 81.5%
accuracy tren tu chua thay trong train: 4935/5754 = 85.8%
  IN : Moi de doa cua cac tran dong dat lon dong vai tro lon trong viec phat trien co so ha tang cua thanh pho.
  OUT: Mỗi để đoạ của các trận động đất lớn động vai trò lớn trong việc phát triển cơ sở hạ tăng của thành phố.
  GOLD: Mối đe dọa của các trận động đất lớn đóng vai trò lớn trong việc phát triển cơ sở hạ tầng của thành phố.
  IN : Trong thoi gian nong nhat trong nam (thang 3 - thang 4), nhiet do thuong dat toi tro len, ngoai tru vung duyen hai, noi gio bien co nhiet do diu buoi chieu.
  OUT: Trong thời gian nóng nhất trong năm (tháng 3 - tháng 4), nhiệt độ thường đất tới trở lên, ngoài trừ vùng duyên hai, nơi giờ biến có nhiệt độ dịu buổi chiều.
  GOLD: Trong thời gian nóng nhất trong năm (tháng 3 - tháng 4), nhiệt độ thường đạt t